# H Permeation Workflow — Part 2

Script-generator + local analysis notebook for the surface→subsurface→permeation pipeline.

**This is the canonical entry point for Part 2.** It writes a standalone orchestrator
(`permeation_run.py`) and a SLURM wrapper (`permeation_run.sh`) that run all heavy
cluster phases automatically. Phase 4 cells below run locally after the cluster jobs finish.

---

**Pipeline overview**

| Phase | Description | Where |
|---|---|---|
| 1 | Hop A NEB — surface H* → subsurface-1 oct | SLURM (GPU + CPU arrays) |
| 2 | Hop B NEB — subsurface-1 → subsurface-2 oct | SLURM (GPU + CPU arrays) |
| 3 | Vibrational frequencies (IS + TS, both hops) | SLURM (CPU array) |
| 4 | TST rate constants at each temperature | Local (fast) |
| 5 | KMC pressure sweeps at each temperature | Local |
| 6 | Richardson-Sieverts permeability (all 3 S₀ options) | Local |

**Inputs** (from Parts 1 and 3):
- `calculation/slabs/slab_relaxed.lammps`
- `calculation/slabs/surface_sites.json`
- `calculation/adsorption/h_atom/h_atom_*_relaxed.lammps` (dedup IS from Part 1)
- `calculation/results/diffusivity_arrhenius.json` (loaded automatically if present)

**Outputs**:
- `calculation/neb_subsurface/` — NEB job trees
- `calculation/results/rate_dict_T{T}K.json` — TST rates per temperature
- `calculation/results/permeation_sweep_T{T}K.json` — KMC sweeps
- `calculation/results/permeability_T{T}K.json` — Φ(T), S(T), J(T)
- `calculation/results/solubility_arrhenius_kmc.json` — Arrhenius S₀ fit

## Cell 1 — Imports & sys.path

In [1]:
import os
import sys
import json

parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from models.config import (
    BASE_DIR,
    MACE_MODEL_ASE,
    SLURM_DEFAULTS,
    N_REPLICAS,
    SPRING_CONST,
    NEB_FTOL,
)
from models.permeation_workflow import (
    generate_permeation_scripts,
    generate_permeation_sh,
    load_barrier_summary,
    load_rate_summary,
    load_kmc_sweeps,
    load_permeability_results,
    plot_barrier_overview,
    plot_mep_overlay,
    plot_kmc_sieverts,
    plot_permeability_vs_T,
    plot_arrhenius_S0,
    plot_bottleneck,
)

print('Imports OK.')

Imports OK.


## Cell 2 — Configuration

Edit all-caps variables below before running.

- Set `DH_DISS_EV` and `DH_ENTRY_EV` **after** the NEB cluster jobs complete and you
  have inspected `delta_E` from the barrier files.
- `D0_M2S` and `E_D_EV` are loaded automatically from `diffusivity_arrhenius.json`
  (written by Part 3) if that file exists; otherwise the placeholders below are used.

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
WORK_DIR           = os.path.join(BASE_DIR, 'calculation')
RELAXED_SLAB_PATH  = os.path.join(WORK_DIR, 'slabs', 'slab_relaxed.lammps')
SURFACE_SITES_JSON = os.path.join(WORK_DIR, 'slabs', 'surface_sites.json')
PHASE2_H_DIR       = os.path.join(WORK_DIR, 'adsorption', 'h_atom')
SUB_NEB_DIR        = os.path.join(WORK_DIR, 'neb_subsurface')
VIB_DIR            = os.path.join(WORK_DIR, 'vibrations')
RESULTS_DIR        = os.path.join(WORK_DIR, 'results')

OUT_PY = os.path.join(WORK_DIR, 'permeation_run.py')
OUT_SH = os.path.join(WORK_DIR, 'permeation_run.sh')

# ── Physical parameters ───────────────────────────────────────────────────────
TEMPERATURES  = [500, 600, 700, 800, 900]    # K
P_VALS_PA     = [1e3, 1e4, 1e5, 5e5, 1e6]   # Pa
A0_M          = 3.52e-10   # m  — Hastelloy N lattice parameter (fallback)
L_M           = 1e-3       # m  — membrane thickness

# ── Diffusivity — loaded from Part 3 if available ────────────────────────────
_diff_json = os.path.join(RESULTS_DIR, 'diffusivity_arrhenius.json')
if os.path.exists(_diff_json):
    with open(_diff_json) as _f:
        _diff = json.load(_f)
    D0_M2S = _diff['D0_m2s']
    E_D_EV = _diff['E_D_eV']
    print(f'Loaded from {_diff_json}:')
    print(f'  D0_M2S = {D0_M2S:.3e} m²/s   E_D_EV = {E_D_EV:.3f} eV')
else:
    D0_M2S = 1.5e-7   # m²/s — placeholder
    E_D_EV = 0.40     # eV   — placeholder
    print(f'WARNING: {_diff_json} not found. Using placeholder D0/ED values.')

# ── NEB thermodynamics — auto-extracted at runtime from NEB results ───────────
# Leave as None — permeation_run.py will auto-extract from ranked_barriers.json
# and rate_dict JSONs. Override here only if auto-extraction fails.
DH_DISS_EV  = None   # eV
DH_ENTRY_EV = None   # eV

# ── KMC parameters ────────────────────────────────────────────────────────────
NX            = 20
NY            = 20
SEED          = 42
KMC_MAX_STEPS = 500_000

# ── NEB parameters ────────────────────────────────────────────────────────────
N_IMAGES      = N_REPLICAS    # 18
SPRING_K      = SPRING_CONST  # 1.0 eV/Å²
NEB_FTOL_VAL  = NEB_FTOL      # 0.05 eV/Å

# ── SLURM configurations ──────────────────────────────────────────────────────
GPU_SLURM_CFG = dict(SLURM_DEFAULTS, partition='multigpu', time='04:00:00')
NEB_SLURM_CFG = dict(SLURM_DEFAULTS, partition='short',
                     gpu=None, cpus_per_task=16, time='12:00:00')
VIB_SLURM_CFG = dict(SLURM_DEFAULTS, partition='short',
                     gpu=None, cpus_per_task=8,  time='06:00:00')

# ── Orchestrator SLURM settings ───────────────────────────────────────────────
ORCH_JOB_NAME      = 'perm_orch'
ORCH_PARTITION     = 'west'
ORCH_CPUS_PER_TASK = 4
ORCH_MEM           = '16G'
ORCH_TIME          = None #'48:00:00'
ORCH_OPENMPI_VER   = SLURM_DEFAULTS.get('openmpi_ver', '')
ORCH_CUDA_VER      = SLURM_DEFAULTS.get('cuda_version', '')
ORCH_CONDA_ENV     = SLURM_DEFAULTS.get('conda_env', 'mace_env')
ORCH_LD_PATHS      = SLURM_DEFAULTS.get('ld_paths', [])

print('Config loaded.')
print(f'  WORK_DIR    : {WORK_DIR}')
print(f'  TEMPERATURES: {TEMPERATURES}')
print(f'  D0_M2S      : {D0_M2S:.3e} m²/s   E_D_EV: {E_D_EV:.3f} eV')
print(f'  DH_DISS_EV  : {DH_DISS_EV}   DH_ENTRY_EV: {DH_ENTRY_EV}  (auto-extracted at runtime)')

Config loaded.
  WORK_DIR    : /projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation
  TEMPERATURES: [500, 600, 700, 800, 900]
  D0_M2S      : 1.500e-07 m²/s   E_D_EV: 0.400 eV
  DH_DISS_EV  : None   DH_ENTRY_EV: None  (auto-extracted at runtime)


## Cell 3 — Generate `permeation_run.py`

Writes the standalone orchestrator script with all configuration injected.  
Open the generated file and verify the injected values before submitting.

In [3]:
generate_permeation_scripts(
    work_dir           = WORK_DIR,
    relaxed_slab_path  = RELAXED_SLAB_PATH,
    surface_sites_json = SURFACE_SITES_JSON,
    phase2_h_dir       = PHASE2_H_DIR,
    sub_neb_dir        = SUB_NEB_DIR,
    vib_dir            = VIB_DIR,
    results_dir        = RESULTS_DIR,
    temperatures       = TEMPERATURES,
    p_vals_pa          = P_VALS_PA,
    a0_m               = A0_M,
    l_m                = L_M,
    d0_m2s             = D0_M2S,
    e_d_ev             = E_D_EV,
    dh_diss_ev         = DH_DISS_EV,
    dh_entry_ev        = DH_ENTRY_EV,
    nx                 = NX,
    ny                 = NY,
    seed               = SEED,
    kmc_max_steps      = KMC_MAX_STEPS,
    gpu_slurm_cfg      = GPU_SLURM_CFG,
    neb_slurm_cfg      = NEB_SLURM_CFG,
    vib_slurm_cfg      = VIB_SLURM_CFG,
    n_images           = N_IMAGES,
    spring_const       = SPRING_K,
    neb_ftol           = NEB_FTOL_VAL,
    out_py             = OUT_PY,
)

FileNotFoundError: [Errno 2] No such file or directory: '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation/permeation_run.py'

## Cell 4 — Generate `permeation_run.sh` and submit

Change `dry_run=True` → `dry_run=False` to actually submit the SLURM job.

In [5]:
from models.create_slurm import submit_slurm_job

generate_permeation_sh(
    orch_job_name      = ORCH_JOB_NAME,
    orch_partition     = ORCH_PARTITION,
    orch_cpus_per_task = ORCH_CPUS_PER_TASK,
    orch_mem           = ORCH_MEM,
    orch_time          = ORCH_TIME,
    orch_openmpi_ver   = ORCH_OPENMPI_VER,
    orch_cuda_version  = ORCH_CUDA_VER,
    orch_conda_env     = ORCH_CONDA_ENV,
    orch_ld_paths      = ORCH_LD_PATHS,
    work_dir           = WORK_DIR,
    out_py             = OUT_PY,
    out_sh             = OUT_SH,
)

# Submit (dry_run=True prints the sbatch command without executing)
jid = submit_slurm_job(OUT_SH, dry_run=True)
print(f'Job ID: {jid}')

FileNotFoundError: [Errno 2] No such file or directory: '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation/permeation_run.sh'

---
## Phase 4 — Local analysis

Run the cells below **after all cluster jobs have completed**.

Each cell calls one analysis function from `models/permeation_workflow.py`.

### 4a. Barrier overview

Loads `neb_barrier.txt` for every converged Hop A and Hop B job and plots
the $E_a$ distribution as a histogram.

In [6]:
import pandas as pd

df_barriers = load_barrier_summary(SUB_NEB_DIR)

if not df_barriers.empty:
    print(df_barriers[['hop', 'sid', 'E_abs', 'E_des', 'delta_E', 'converged']]
          .to_string(index=False))
    plot_barrier_overview(df_barriers, SUB_NEB_DIR)
else:
    print('No barrier data found yet — run cluster phases first.')

TypeError: unsupported operand type(s) for |: 'types.GenericAlias' and 'NoneType'

### 4b. MEP overlay

Overlays all NEB minimum energy paths for Hop A and Hop B.

In [ ]:
plot_mep_overlay(SUB_NEB_DIR)

### 4c. TST rate summary

Loads `rate_dict_T{T}K.json` for every temperature and shows $k(T)$
as a semi-log Arrhenius plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

df_rates = load_rate_summary(RESULTS_DIR, TEMPERATURES)

if df_rates.empty:
    print('No rate data found yet.')
else:
    print(df_rates.head(10).to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 5))
    for label, grp in df_rates.groupby('label'):
        grp = grp.sort_values('T_K')
        ax.semilogy(grp['T_K'], grp['k_forward'], 'o-', lw=1.2, alpha=0.8, label=label)
    ax.set_xlabel('Temperature  [K]')
    ax.set_ylabel('$k_{fwd}$  [s$^{-1}$]')
    ax.set_title('Forward rate constants vs temperature')
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'rate_vs_T.png'), dpi=150)
    plt.show()

### 4d. KMC Sieverts check

Plots $J$ vs $\sqrt{P}$ at each temperature. A linear relationship confirms
Sieverts' law and bulk-diffusion-limited transport.

In [ ]:
plot_kmc_sieverts(RESULTS_DIR, TEMPERATURES)

### 4e. Permeability Φ(T)

Plots $\Phi(T)$ and $\log_{10}(\Phi)$ vs $1000/T$ for all three $S_0$ options:
- Option 1 — lattice site density
- Option 2 — TST detailed balance
- Option 3 — KMC empirical Sieverts fit

In [ ]:
plot_permeability_vs_T(RESULTS_DIR, TEMPERATURES)

### 4f. Arrhenius S₀ fit (Option 3)

Reads `solubility_arrhenius_kmc.json` and overlays the Arrhenius $S_0$ fit
on $\Phi(T)$, reporting $\Delta H_{sol}^{KMC}$ and $R^2$.

In [ ]:
plot_arrhenius_S0(RESULTS_DIR)

### 4g. Bottleneck diagnosis

Bar chart of $R^2$ (J vs $\sqrt{P}$) at each temperature.

- $R^2 \geq 0.98$ → bulk diffusion is rate-limiting (Sieverts' law holds)
- $R^2 < 0.98$ → surface kinetics (dissociation / recombination) are the bottleneck

In [ ]:
plot_bottleneck(RESULTS_DIR, TEMPERATURES)